### Assignment 1

Construct a Gaussian mixture model class using only numpy

In [1]:
import numpy as np

In [2]:
class GaussianMixture:
    def __init__(self, n_components, max_iter=100, tol=1e-6, random_state=None):
        self.K = n_components # K Gaussian components
        self.max_iter = max_iter # Maximum number of iterations
        self.tol = tol # Tolerance for convergence 
        self.random_state = random_state # Random seed for reproducibility
    
    def _initialize_parameters(self, X):
        N, D = X.shape # Number of samples and dimensions
        np.random.seed(self.random_state)

        # Initialize centroids randomly from the data
        self.mu = X[np.random.choice(N, self.K, replace=False)] # Randomly select K samples as initial means

        # Initialize covariance matrics as identity matrics
        self.sigma = np.array([np.eye(D) for _ in range(self.K)]) # Identity matrices D x D for each component

        # Initialize mixing coefficients uniformly
        self.pi = np.full(self.K, 1 / self.K) # Mixing coefficient pi is uniform across components and sums to 1

    # Computes the multivariate Gaussian probability density function
    def _multivariate_gaussian(self, x, mean, cov):
        # Compute the determinant and inverse of the covariance matrix
        D = x.shape[0]
        cov_det = np.linalg.det(cov) # determinant of covariance matrix
        cov_inv = np.linalg.inv(cov) # inverse of covariance matrix

        # Compute the normalization constant
        norm_const = 1  / np.sqrt((2 * np.pi) ** D * cov_det) # Ensure the total probability integrates to 1

        # Compute the exponent term
        diff = x - mean # Difference vector
        exponent = -0.5 * diff.T @ cov_inv @ diff # Mahalanobis distance squared
        return norm_const * np.exp(exponent)
    
    # E-step: Calculate responsibilities (probabilities of each component given the data)
    # gamma[n, k] is the probability that sample n belongs to component k
    def _e_step(self, X):
        N = X.shape[0]
        self.gamma = np.zeros((N, self.K)) # responsibility matrix
        for n in range(N):
            for k in range(self.K):
                self.gamma[n, k] = self.pi[k] * self._multivariate_gaussian(X[n], self.mu[k], self.sigma[k])
            self.gamma[n] /= np.sum(self.gamma[n]) # Normalize
    
    # Update parameters using the current responsibilities
    def _m_step(self, X):
        N, D = X.shape
        N_k = np.sum(self.gamma, axis=0) 

        # Update mu
        self.mu = (self.gamma.T @ X) / N_k[:, np.newaxis]

        # Update sigma
        self.sigma = np.zeros((self.K, D, D))
        for k in range(self.K):
            for n in range(N):
                diff = X[n] - self.mu[k]
                self.sigma[k] += self.gamma[n, k] * np.outer(diff, diff)
            self.sigma[k] /= N_k[k]
        
        # Update pi
        self.pi = N_k / N

    def _compute_log_likelihood(self, X):
        N = X.shape[0]
        log_likelihood = 0

        for n in range(N):
            prob = 0
            for k in range(self.K):
                prob += self.pi[k] * self._multivariate_gaussian(X[n], self.mu[k], self.sigma[k])
            log_likelihood += np.log(prob + 1e-10)  # Adding a small constant to avoid log(0)
        
        return log_likelihood
    
    def fit(self, X):
        self._initialize_parameters(X)

        log_likelihood_old = None
        for iteration in range(self.max_iter):
            self._e_step(X)
            self._m_step(X)

            log_likelihood = self._compute_log_likelihood(X)

            if log_likelihood_old is not None and abs(log_likelihood - log_likelihood_old) < self.tol:
                break
            log_likelihood_old = log_likelihood

        self.log_likelihood = log_likelihood

    def predict_proba(self, X):
        N = X.shape[0]
        probs = np.zeros((N, self.K))
        for n in range(N):
            for k in range(self.K):
                probs[n, k] = self.pi[k] * self._multivariate_gaussian(X[n], self.mu[k], self.sigma[k])
            probs[n] /= np.sum(probs[n])  # Normalize
        return probs
    
    def predict(self, X):
        return np.argmax(self.predict(X), axis=1)

In [3]:
import cv2

In [4]:
def segment_background(image_path, n_components=2):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB
    h, w, c = img.shape
    pixels = img_rgb.reshape(-1, 3).astype(np.float64) # Reshape to (N, 3)

    gmm = GaussianMixture(n_components=n_components, max_iter=20, random_state=42)
    gmm.fit(pixels)
    labels = gmm.predict(pixels)

    counts = np.bincount(labels)
    background_label = np.argmax(counts)  # Most frequent label is considered background

    mask = (labels != background_label).astype(np.uint8) * 255
    mask = mask.reshape(h, w)  # Reshape back to original image dimensions

    # Apply mask to the original image
    foreground = cv2.bitwise_and(img, img, mask=mask)
    return foreground, mask

image_path = 'cow.jpg'
foreground, mask = segment_background(image_path)

# Save the results
cv2.imwrite('foreground.jpg', foreground)
cv2.imwrite('mask.jpg', mask)

C:\Users\AcerI5\AppData\Local\Temp\ipykernel_18476\852906686.py:44: RuntimeWarning: invalid value encountered in divide
  self.gamma[n] /= np.sum(self.gamma[n]) # Normalize
C:\Users\AcerI5\AppData\Roaming\Python\Python311\site-packages\numpy\linalg\linalg.py:2180: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


RecursionError: maximum recursion depth exceeded